# Módulo 1. Tarefa 1. Jogo do galo com minimax

# Parte 1. Planeamento

O problema escolhido é o __jogo do galo__ (tabuleiro 3 por 3, jogadores X e O). A abordagem de IA é __baseada em pesquisa__, com a técnica __minimax__. A descrição completa do problema e da abordagem está no PDF de entrega.

No programa, as casas estão numeradas de 1 a 9:

```
 1 | 2 | 3
---+---+---
 4 | 5 | 6
---+---+---
 7 | 8 | 9
```

# Parte 2. Implementação

#### Preparação: símbolos, linhas vencedoras e funções do tabuleiro

O tabuleiro é uma lista com 9 elementos, um por casa. A casa número 1 está na posição 0 da lista, a casa 2 na posição 1, e assim sucessivamente (por isso, o código usa sempre `numero_casa - 1`).

In [ ]:
# Importa o módulo csv, para criar e ler o ficheiro com os tabuleiros de teste
import csv

# Símbolos usados no jogo
simbolo_casa_vazia = "-"
simbolo_jogador_x = "X"
simbolo_jogador_o = "O"

# As 8 linhas vencedoras (linhas, colunas e diagonais), com as casas numeradas de 1 a 9
linhas_vencedoras = [
    (1, 2, 3), (4, 5, 6), (7, 8, 9),  # linhas
    (1, 4, 7), (2, 5, 8), (3, 6, 9),  # colunas
    (1, 5, 9), (3, 5, 7),             # diagonais
]


def mostrar_tabuleiro(tabuleiro):
    # Mostra o tabuleiro em 3 linhas de 3 casas
    for inicio_linha in (0, 3, 6):
        print(" ".join(tabuleiro[inicio_linha:inicio_linha + 3]))


def calcular_vencedor(tabuleiro):
    # Devolve "X" ou "O" se alguém tiver três símbolos em linha; se não, devolve None
    for casa_a, casa_b, casa_c in linhas_vencedoras:
        simbolo_a = tabuleiro[casa_a - 1]
        if simbolo_a != simbolo_casa_vazia and simbolo_a == tabuleiro[casa_b - 1] == tabuleiro[casa_c - 1]:
            return simbolo_a
    return None


def listar_casas_livres(tabuleiro):
    # Devolve os números (1 a 9) das casas que ainda estão vazias
    casas_livres = []
    for numero_casa in range(1, 10):
        if tabuleiro[numero_casa - 1] == simbolo_casa_vazia:
            casas_livres.append(numero_casa)
    return casas_livres


def obter_adversario(jogador):
    # Devolve o símbolo do outro jogador
    if jogador == simbolo_jogador_x:
        return simbolo_jogador_o
    return simbolo_jogador_x


print("Funções do tabuleiro definidas com sucesso.")

#### A técnica de pesquisa: minimax

A função `minimax` devolve a pontuação de um tabuleiro do ponto de vista do computador (+1 se ganha, 0 se empata, -1 se perde), admitindo que os dois jogadores jogam o melhor possível: o computador escolhe a maior pontuação e o adversário a menor. Chama-se a si própria (é __recursiva__) para cada jogada possível, sobre uma cópia da lista, para não estragar o tabuleiro original. O dicionário `estatisticas_pesquisa` só conta quantas posições são avaliadas.

In [ ]:
# Dicionário para contar quantas posições o minimax avalia (usado na Parte 3, na demonstração)
estatisticas_pesquisa = {"posicoes_avaliadas": 0}


def minimax(tabuleiro, jogador_a_jogar, simbolo_computador):
    # Devolve a pontuação do tabuleiro para o computador: +1 se ganha, 0 se empata, -1 se perde,
    # admitindo que os dois jogadores jogam o melhor possível
    estatisticas_pesquisa["posicoes_avaliadas"] += 1

    # Casos de paragem: o jogo já acabou (vitória de alguém ou tabuleiro cheio)
    vencedor = calcular_vencedor(tabuleiro)
    if vencedor == simbolo_computador:
        return 1
    if vencedor is not None:
        return -1
    casas_livres = listar_casas_livres(tabuleiro)
    if len(casas_livres) == 0:
        return 0

    # Experimenta cada jogada possível e pede a pontuação do tabuleiro resultante (recursão)
    pontuacoes_possiveis = []
    for numero_casa in casas_livres:
        tabuleiro_seguinte = tabuleiro.copy()
        tabuleiro_seguinte[numero_casa - 1] = jogador_a_jogar
        pontuacao = minimax(tabuleiro_seguinte, obter_adversario(jogador_a_jogar), simbolo_computador)
        pontuacoes_possiveis.append(pontuacao)

    # Na vez do computador escolhe-se a maior pontuação; na vez do adversário, a menor
    if jogador_a_jogar == simbolo_computador:
        return max(pontuacoes_possiveis)
    return min(pontuacoes_possiveis)


def calcular_pontuacao_de_cada_jogada(tabuleiro, jogador_a_jogar):
    # Devolve um dicionário {número da casa: pontuação} com todas as jogadas possíveis
    pontuacoes_por_casa = {}
    for numero_casa in listar_casas_livres(tabuleiro):
        tabuleiro_seguinte = tabuleiro.copy()
        tabuleiro_seguinte[numero_casa - 1] = jogador_a_jogar
        pontuacoes_por_casa[numero_casa] = minimax(
            tabuleiro_seguinte, obter_adversario(jogador_a_jogar), jogador_a_jogar
        )
    return pontuacoes_por_casa


def escolher_melhor_jogada(tabuleiro, jogador_a_jogar):
    # Devolve o número da casa com a melhor pontuação; se houver empate, fica com a primeira
    pontuacoes_por_casa = calcular_pontuacao_de_cada_jogada(tabuleiro, jogador_a_jogar)
    melhor_pontuacao = max(pontuacoes_por_casa.values())
    for numero_casa, pontuacao in pontuacoes_por_casa.items():
        if pontuacao == melhor_pontuacao:
            return numero_casa


print("Funções do minimax definidas com sucesso.")

#### O conjunto de dados: criar o ficheiro CSV

Cria-se o ficheiro __"tabuleiros_teste.csv"__ com 10 tabuleiros fictícios. Cada tabuleiro tem 9 símbolos (os três primeiros são a linha de cima, os três seguintes a do meio e os três últimos a de baixo; o `-` é uma casa vazia). A jogada esperada foi escolhida à mão.

In [ ]:
# Cria o ficheiro tabuleiros_teste.csv com os tabuleiros de teste
cabecalho_tabuleiros = ["descricao", "tabuleiro", "jogador_a_jogar", "jogada_esperada"]
dados_tabuleiros = [
    ["Vitória imediata (o adversário também ameaçava ganhar)", "XX-OO----", "X", 3],
    ["Bloquear a linha de cima", "XX--O----", "O", 3],
    ["Bloquear a coluna da esquerda", "X---O-X--", "O", 4],
    ["Bloquear a coluna do meio", "XO--O---X", "X", 8],
    ["Responder a uma abertura em canto", "X--------", "O", 5],
    ["Vitória imediata do O (em vez de bloquear)", "XX-OO-X--", "O", 6],
    ["Vitória na diagonal", "OXXXOO-X-", "O", 9],
    ["Vitória na linha do meio", "X---OOX-X", "O", 4],
    ["Vitória na coluna da direita (que também bloqueia)", "O--XOXO-X", "X", 3],
    ["Criar um duplo ataque", "XXOO-----", "X", 5],
]

with open("tabuleiros_teste.csv", "w", newline="", encoding="utf-8") as ficheiro_csv:
    escritor = csv.writer(ficheiro_csv)
    escritor.writerow(cabecalho_tabuleiros)
    escritor.writerows(dados_tabuleiros)

print("Ficheiro tabuleiros_teste.csv criado com sucesso.")

# Verificação: lê novamente o ficheiro criado, para confirmar o conteúdo
with open("tabuleiros_teste.csv", "r", encoding="utf-8") as ficheiro_csv:
    print(ficheiro_csv.read())

#### Testar o protótipo com os dados

Para cada tabuleiro do ficheiro CSV, compara-se a jogada escolhida pelo minimax com a jogada esperada.

In [ ]:
# Lê o ficheiro tabuleiros_teste.csv, em formato dicionário
with open("tabuleiros_teste.csv", "r", encoding="utf-8") as ficheiro_csv:
    leitor = csv.DictReader(ficheiro_csv)
    tabuleiros_teste = list(leitor)

numero_acertos = 0

# Para cada tabuleiro de teste, calcula a jogada com o minimax e compara com a esperada
for numero_teste, caso_teste in enumerate(tabuleiros_teste, start=1):
    tabuleiro_atual = list(caso_teste["tabuleiro"])  # converte o texto numa lista de 9 símbolos
    jogador_a_jogar = caso_teste["jogador_a_jogar"]
    jogada_esperada = int(caso_teste["jogada_esperada"])

    jogada_do_minimax = escolher_melhor_jogada(tabuleiro_atual, jogador_a_jogar)

    if jogada_do_minimax == jogada_esperada:
        resultado = "CORRETO"
        numero_acertos += 1
    else:
        resultado = "ERRADO"

    print(f"Teste {numero_teste}: {caso_teste['descricao']}")
    print(f"    Joga o {jogador_a_jogar} | esperada: casa {jogada_esperada} | minimax: casa {jogada_do_minimax} -> {resultado}")

print(f"\nResultado: {numero_acertos} acertos em {len(tabuleiros_teste)} tabuleiros de teste.")

# Parte 3. Demonstração e análise

#### Exemplos de entrada e saída

A __entrada__ é um tabuleiro e o jogador que vai jogar. A __saída__ é a pontuação de cada jogada possível e a casa escolhida (a de melhor pontuação).

In [ ]:
def mostrar_exemplo(descricao, texto_tabuleiro, jogador_a_jogar):
    # Mostra a entrada (tabuleiro e jogador) e a saída (pontuações, jogada escolhida e tabuleiro final)
    tabuleiro_exemplo = list(texto_tabuleiro)
    print(f"--- {descricao} ---")
    print("ENTRADA: tabuleiro atual")
    mostrar_tabuleiro(tabuleiro_exemplo)
    print(f"Joga o {jogador_a_jogar}.")

    # Escolhe a jogada, contando as posições avaliadas pela pesquisa
    estatisticas_pesquisa["posicoes_avaliadas"] = 0
    casa_escolhida = escolher_melhor_jogada(tabuleiro_exemplo, jogador_a_jogar)
    posicoes_avaliadas = estatisticas_pesquisa["posicoes_avaliadas"]

    pontuacoes_por_casa = calcular_pontuacao_de_cada_jogada(tabuleiro_exemplo, jogador_a_jogar)
    print(f"SAÍDA: pontuação de cada jogada possível (casa: pontuação) = {pontuacoes_por_casa}")
    print(f"Jogada escolhida: casa {casa_escolhida}")
    print(f"Posições avaliadas na pesquisa: {posicoes_avaliadas:,}".replace(",", " "))

    tabuleiro_exemplo[casa_escolhida - 1] = jogador_a_jogar
    print("Tabuleiro depois da jogada:")
    mostrar_tabuleiro(tabuleiro_exemplo)
    print()


# Exemplo 1: o computador (O) tem de bloquear a linha de cima do adversário
mostrar_exemplo("Exemplo 1: bloquear o adversário", "XX--O----", simbolo_jogador_o)

# Exemplo 2: o computador (X) ganha ao completar a linha de cima
mostrar_exemplo("Exemplo 2: ganhar o jogo", "XX-OO----", simbolo_jogador_x)

# Exemplo 3: o computador (O) responde a uma abertura do X num canto
mostrar_exemplo("Exemplo 3: responder a uma abertura em canto", "X--------", simbolo_jogador_o)

# Exemplo 4: o computador (X) joga primeiro, com o tabuleiro vazio (a pesquisa maior possível)
mostrar_exemplo("Exemplo 4: tabuleiro vazio", "---------", simbolo_jogador_x)

#### Jogar contra o computador

Tu és o __X__ e jogas primeiro; o computador é o __O__. Em cada jogada, escreve o número da casa (de 1 a 9).

In [ ]:
def pedir_jogada_ao_humano(tabuleiro):
    # Pede ao jogador o número de uma casa livre, até receber uma resposta válida
    casas_livres = listar_casas_livres(tabuleiro)
    while True:
        texto_digitado = input(f"Insere o número da casa onde queres jogar {casas_livres}: ")
        if texto_digitado.isdigit() and int(texto_digitado) in casas_livres:
            return int(texto_digitado)
        print("Jogada inválida. Escolhe uma das casas livres.")


def jogar_contra_o_computador():
    # Joga uma partida completa: o humano é o X e começa; o computador é o O
    tabuleiro_jogo = [simbolo_casa_vazia] * 9
    simbolo_humano = simbolo_jogador_x
    simbolo_computador = simbolo_jogador_o
    jogador_a_jogar = simbolo_humano

    print("Jogo do galo: tu és o X e jogas primeiro; o computador é o O.")
    print("Casas numeradas de 1 a 9, da esquerda para a direita e de cima para baixo.\n")

    # A partida continua enquanto não houver vencedor e ainda houver casas livres
    while calcular_vencedor(tabuleiro_jogo) is None and len(listar_casas_livres(tabuleiro_jogo)) > 0:
        if jogador_a_jogar == simbolo_humano:
            casa_escolhida = pedir_jogada_ao_humano(tabuleiro_jogo)
        else:
            casa_escolhida = escolher_melhor_jogada(tabuleiro_jogo, simbolo_computador)
            print(f"O computador joga na casa {casa_escolhida}.")

        tabuleiro_jogo[casa_escolhida - 1] = jogador_a_jogar
        mostrar_tabuleiro(tabuleiro_jogo)
        print()
        jogador_a_jogar = obter_adversario(jogador_a_jogar)

    # Mostra o resultado final
    vencedor = calcular_vencedor(tabuleiro_jogo)
    if vencedor == simbolo_humano:
        print("Ganhaste!")
    elif vencedor == simbolo_computador:
        print("O computador ganhou.")
    else:
        print("Empate.")


# Executa uma partida (escreve as jogadas na caixa de texto que aparece no topo do VS Code)
jogar_contra_o_computador()

#### Exemplo de uma limitação

O __X__ pode ganhar já na casa 9, mas várias jogadas têm pontuação +1 e o programa fica com a primeira.

In [ ]:
# Exemplo da limitação: o X ganha logo na casa 9, mas há outras jogadas com a mesma pontuação
mostrar_exemplo("Limitação: vitória imediata vs. vitória mais tarde", "XOO-X----", simbolo_jogador_x)